# K-Phase: Knowledge Transfer (Deployment)

## Ziel
Die K-Phase überführt das ausgewählte Modell aus der Analyse in eine betreibbare Lösung mit dokumentierter Übergabe.

Im Fokus stehen dabei nicht nur die reine Bereitstellung des Modells, sondern auch die Fragen, wie die Lösung im Betrieb genutzt wird, wie Wissen im Team gesichert bleibt und welche Voraussetzungen für Wartung, Weiterentwicklung und Reproduzierbarkeit erfüllt sein müssen.

Die K-Phase beantwortet damit insbesondere vier praktische Übergabefragen:
- Wie wird das Modell technisch bereitgestellt?
- Welche Artefakte, Dateien und Dokumente müssen für eine Übergabe vorliegen?
- Wie kann ein neues Teammitglied oder ein externer Prüfer das Projekt nachvollziehen und starten?
- Welche Betriebsrisiken, Grenzen und nächsten Ausbaustufen sind bereits bekannt?


## Deployment-Strategie

Für das Projekt wurden zwei realistische Bereitstellungswege betrachtet. Beide sind technisch sinnvoll, unterscheiden sich aber deutlich hinsichtlich Komplexität, Betriebsaufwand und Eignung für Demo, Lehre und späteren Produktivbetrieb.

### Option 1: Embedded Deployment (Streamlit)
- Modell wird direkt in der App geladen.
- Vorteil: schnell umsetzbar, geringe Infrastruktur.
- Vorteil: besonders geeignet für Demo, Lehre, PoC und prüfbare Abgabe.
- Nachteil: begrenzte Skalierbarkeit.
- Nachteil: UI, Inferenz und Datenlogik laufen enger gekoppelt in einem Prozess.
- Nachteil: bei steigender Last oder mehreren Nutzern wird die Architektur schneller unübersichtlich.

### Option 2: Model-as-a-Service (FastAPI)
- Modell als separater Inferenzservice (`/predict`).
- Vorteil: entkoppelte Architektur, bessere Skalierbarkeit und Wiederverwendbarkeit.
- Vorteil: saubere Trennung zwischen Frontend, Inferenz, Datenaufnahme und Persistenz.
- Vorteil: besser geeignet für spätere Systemintegration, Authentifizierung und Monitoring.
- Nachteil: höherer Betriebs- und Monitoringaufwand.
- Nachteil: für eine reine Uni-Demo oft unnötig komplex, wenn keine dauerhafte Infrastruktur vorhanden ist.

## Betriebsbewertung
- **Für die Bewertung an der Hochschule** ist Embedded Deployment die robusteste Variante, weil keine zusätzliche Betriebsumgebung, keine Schlüsselverwaltung und kein externer API-Zugang zwingend nötig sind.
- **Für ein Teamprojekt mit Weiterentwicklung** ist die FastAPI-Variante das bessere Zielbild, da sie fachliche Verantwortlichkeiten klarer trennt.
- **Für den Übergang zwischen beiden Welten** ist eine hybride Strategie sinnvoll: dieselben Daten- und Modellartefakte werden zunächst in Streamlit verwendet und später in einen Service extrahiert.

## Empfohlenes Zielbild
Kurzfristig Embedded für Demo/PoC, mittelfristig Migration auf FastAPI-Inferenzservice mit Streamlit als Client.

Das bedeutet konkret:
- Phase 1: reproduzierbare lokale Ausführung mit Streamlit, Docker und Repository-Artefakten.
- Phase 2: Stabilisierung der Schnittstellen und Auslagerung einzelner Funktionen in das Backend.
- Phase 3: optionaler Produktivpfad mit separater Inferenz, Datenbankbetrieb, Monitoring und kontrollierter Datenaufnahme.


## K-Check: Deployment und Repository-Übergabe

- **Die fertige Streamlit App**
  Übergabe erfolgt über die App im Repository (`src/traffic_app`) inklusive Modellartefakt und Reports.
- **Das GitHub Repo**
  Zentrale Projektquelle: https://github.com/lukasp1209/Traffic-Prediction-Optimization
- **Wissensbasis im Repo**
  Das Repo enthält reproduzierbare Notebook-Phasen (Q/U/A/C/K), Exportartefakte unter `reports/` sowie Dokumentation unter `docs/`.

## Übergabepaket im Detail
- **Quellcode**
  Enthalten sind Streamlit-Frontend, Backend-Module, Datenlogik, Forecasting, Kartenlogik und Optimierungslogik.
- **Modellartefakte**
  Das beste Modell, Metriken und Testvorhersagen liegen in exportierbarer Form vor und können unabhängig von den Notebooks weiterverwendet werden.
- **Notebook-Dokumentation**
  Die Projektlogik ist entlang der Phasen Q/U/A/C/K nachvollziehbar dokumentiert. Dadurch bleibt erkennbar, wie aus der Fragestellung eine lauffähige Lösung entstanden ist.
- **Betriebsnahe Dokumentation**
  README, Model Card und ergänzende Leitfäden beschreiben Start, Nutzung, Konfiguration, Grenzen und Erweiterungsmöglichkeiten.

## Was ein Prüfer oder neues Teammitglied damit tun kann
- Das Repository klonen und die Anwendung lokal starten.
- Die Modellwahl anhand der gespeicherten Leistungswerte nachvollziehen.
- Die Streamlit-App mit lokalen Daten ohne externe Dienste demonstrieren.
- Die Projektentscheidungen anhand der Notebook-Phasen, Reports und Dokumentation fachlich prüfen.

## Bekannte Betriebsgrenzen
- Externe Live-APIs sind für den Produktivbetrieb vorgesehen, für die Hochschulbewertung aber bewusst durch Demo-/Offline-Modi ersetzbar.
- Kartenabfragen über OSM/Overpass können je nach Umfang langsam sein; deshalb existiert ein Schnellmodus mit lokal bevorzugten Geometrien.
- Das aktuelle Deployment ist auf Transparenz und Reproduzierbarkeit optimiert, nicht auf Hochlastbetrieb.

## Empfohlene nächste Ausbaustufen
- Ergänzung eines echten `/predict`-Endpunkts für Model-as-a-Service.
- Trennung von Trainings- und Inferenzpfad in getrennte Runtime-Komponenten.
- Monitoring für Datenqualität, Prognosefehler und Antwortzeiten.
- Versionierung von Modellartefakten und Datenschnitten für saubere Re-Trainings.


In [1]:
from pathlib import Path
import json
import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORT_DIR = repo_root / "reports"
MODEL_DIR = REPORT_DIR / "models"
summary_path = REPORT_DIR / "model_summary.json"
results_path = REPORT_DIR / "model_results.csv"

missing = [str(p) for p in [summary_path, results_path] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "A-Phase-Ergebnisse fehlen. Bitte zuerst A-Phase_Algorithms.ipynb ausführen. Fehlend: " + ", ".join(missing)
    )

summary = json.loads(summary_path.read_text(encoding="utf-8"))
results = pd.read_csv(results_path).sort_values("RMSE").reset_index(drop=True)
best_model = summary["best_model"]

artifact_candidates = list(MODEL_DIR.glob(f"best_model_{best_model}.pkl"))
artifact_exists = len(artifact_candidates) > 0

portfolio_summary = {
    "Project": "Traffic Prediction & Optimization",
    "Methodology": "QUA3CK",
    "Best_Algorithm": best_model,
    "Dataset_Rows": int(summary["dataset_rows"]),
    "Train_Rows": int(summary["n_train"]),
    "Test_Rows": int(summary["n_test"]),
    "Top_Result": results.iloc[0].to_dict(),
    "Model_Artifact_Available": artifact_exists,
    "Deployment_Target": "FastAPI Inference Service + Streamlit UI",
    "Next_Steps": [
        "API endpoint /predict mit Request-Validierung",
        "Containerized deployment via docker compose",
        "Monitoring: Latenz, Fehlerrate, RMSE-Drift",
        "Regelmäßiges Re-Training mit aktuellen Verkehrsdaten",
    ],
}

print("K-Phase Portfolio Summary")
print("=" * 60)
for key, value in portfolio_summary.items():
    print(f"{key}: {value}")


K-Phase Portfolio Summary
Project: Traffic Prediction & Optimization
Methodology: QUA3CK
Best_Algorithm: RandomForest
Dataset_Rows: 8734
Train_Rows: 6987
Test_Rows: 1747
Top_Result: {'Model': 'RandomForest', 'MAE': 4874.081155802304, 'RMSE': 6807.078467974301, 'R2': 0.9597666764999648}
Model_Artifact_Available: True
Deployment_Target: FastAPI Inference Service + Streamlit UI
Next_Steps: ['API endpoint /predict mit Request-Validierung', 'Containerized deployment via docker compose', 'Monitoring: Latenz, Fehlerrate, RMSE-Drift', 'Regelmäßiges Re-Training mit aktuellen Verkehrsdaten']


In [2]:
from textwrap import dedent

model_card = dedent(f"""
# Model Card (Kurzfassung)

## Modell
- Name: {summary['best_model']}
- Anwendungsfall: Kurzfristige Verkehrsprognose (Stundenebene)

## Trainingskontext
- Datensätze: {summary['dataset_rows']}
- Train/Test: {summary['n_train']} / {summary['n_test']} (chronologisch)

## Kernmetriken (Test)
- MAE: {results.iloc[0]['MAE']:.3f}
- RMSE: {results.iloc[0]['RMSE']:.3f}
- R2: {results.iloc[0]['R2']:.4f}

## Grenzen
- Synthetische Anteile möglich (abhängig von U-Phase-Konfiguration)
- Externe Ereignisse (Unfälle, Großevents, Baustellen) nur begrenzt abgebildet
- Regelmäßige Validierung auf Live-Daten erforderlich
""").strip()

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
model_card_path = repo_root / "docs" / "guides" / "model-card.md"

# Fehlende Verzeichnisse robust anlegen
model_card_path.parent.mkdir(parents=True, exist_ok=True)

# Datei schreiben
model_card_path.write_text(model_card, encoding="utf-8")

print(f"Model Card geschrieben: {model_card_path.resolve()}")
print(f"Aktueller Arbeitsordner: {Path.cwd()}")


Model Card geschrieben: Z:\Traffic-Prediction-Optimization\docs\guides\model-card.md
Aktueller Arbeitsordner: Z:\Traffic-Prediction-Optimization\notebooks


## Ergebnis der K-Phase
- Übergabefähige Zusammenfassung wurde erzeugt.
- Modellartefakt und Leistungsdaten sind dokumentiert und maschinenlesbar abgelegt.
- Ein operativer Deployment-Pfad ist definiert (Streamlit + Docker + Repository-Übergabe über GitHub).
- Die Lösung ist für Demo und Bewertung ohne externe API-Schlüssel betreibbar.
- Das Projekt enthält damit nicht nur ein trainiertes Modell, sondern ein vollständiges Übergabepaket aus Code, Artefakten, Dokumentation und Betriebslogik.

### Management-Fazit
Die K-Phase zeigt, dass das Projekt nicht auf der Analyseebene stehen bleibt. Das Ergebnis ist eine nachvollziehbare, demonstrierbare und erweiterbare Lösung, die bereits heute als Entscheidungsunterstützung präsentiert werden kann und gleichzeitig eine klare technische Perspektive für eine spätere Professionalisierung besitzt.
